# Day 1 · Module 5: Agent Teams (Orchestration, Disagreement Rule, Synthesizer)

**Objective:** stand up a small pipeline of read-only and write-scoped subagents behind one orchestrating command, and build a synthesizer that decides — SHIP, REVISE, or ESCALATE — instead of just summarizing what the team produced.


## 1 · The idea

A single agent working alone has one point of view. An agent *team* fans a task out to specialists with narrow, non-overlapping jobs — an explorer maps the codebase, an implementer writes code, a tester writes and runs tests, a critic attacks the result — and then needs something to bring it back together. That something is not a summary. Summarizing "explorer said X, critic said Y" restates disagreement without resolving it.

The orchestrator's job is to run the team in order, pass artifacts forward, and enforce a disagreement rule: if the critic raises a blocker, the implementer gets exactly one revision loop to address it, and if the blocker is still open after that loop, the pipeline escalates rather than looping forever or waving it through. The synthesizer's job is to read everything the team produced and *decide* — with evidence, with the risks it is choosing to accept, and with an explicit statement of how confident it is in that decision.


### The roster

One line per agent. Each job is narrow and does not overlap the others —
that's what makes them a *team* instead of one agent role-playing six parts:

- **Orchestrator** — runs the team in order, passes artifacts forward, enforces the disagreement rule.
- **Explorer** — maps the posting/webhook path; writes nothing.
- **Implementer** — makes the one wiring change; owns the revision loop.
- **Tester** — writes and runs tests against the change.
- **Critic** — attacks the change; emits BLOCKER / CLEAR only.
- **Synthesizer** — reads every artifact and DECIDES SHIP / REVISE / ESCALATE with evidence, accepted risk, and explicit confidence.


### Grounding

`scenarios/m5-webhooks.md` asks for `webhooks.deliver()` to be wired into the posting path so a webhook fires on `transfer.posted`. `src/contoso/webhooks.py` shows `deliver()` is deliberately naive: synchronous, in order, with no timeout, no retry, and no isolation between subscribers. `src/contoso/posting.py` shows the posting path `post()` already commits the ledger entry as a separate step from authorizing the transfer against its hold — the two are not wrapped in a single transaction.

Wiring `deliver()` in naively means: a slow or failing merchant endpoint can block or fail the posting itself, or — if a caller retries on failure — the same posting gets reported to a subscriber more than once, since `deliver()` has no idempotency or delivery-tracking of its own. A double-delivered payment notification can trigger duplicate downstream fulfillment for the merchant; a lost one means the merchant never learns the payment settled. An honest pipeline surfaces this as a blocker and, if it isn't resolved by isolating delivery from the posting path, ESCALATEs. Forcing a green run by shipping anyway would mean losing or double-delivering payment notifications in production — the synthesizer's job is to refuse that outcome, not paper over it.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` for the grounding check, the exercise, and the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


With `proj` resolved, confirm the shape of the naive webhook and the posting path directly against the real source:


In [ ]:
import pathlib
src = pathlib.Path(proj) / "src" / "contoso"
wh = (src / "webhooks.py").read_text()
post = (src / "posting.py").read_text()
print("webhooks.deliver() is synchronous, no isolation:", "def deliver" in wh and "synchronous" in wh)
print("posting.post() is the posting path deliver() would be wired into:", "def post" in post and "ledger.record_entry" in post)
print("event name this ticket wires deliver() to:", "transfer.posted")


### The disagreement rule, as a spec

This is the exact logic you encode into `run-pipeline.md` — not prose to paraphrase, but a rule to implement precisely:

```text
on critic verdict:
  BLOCKER  -> implementer gets exactly 1 revision loop
    still BLOCKER after that loop -> ESCALATE (never loop again, never force-green)
    cleared -> continue to synthesizer
  CLEAR    -> continue to synthesizer
on synthesizer confidence < threshold -> HALT / ESCALATE (do not SHIP)
```

Two failure modes this prevents: a critic and implementer arguing forever in an unbounded loop, and a pipeline that quietly forces a green result when nobody is actually confident in it. One revision loop, then a decision — SHIP, REVISE, or ESCALATE — is always reached.


### Summarizing vs. deciding

The whole point of a synthesizer is that it does not read like a `cat` of the upstream artifacts. Same inputs, two very different outputs:

| Summarizing synthesizer (wrong) | Deciding synthesizer (right) |
|---|---|
| "Explorer mapped the posting path; critic flagged the webhook ordering; tests pass; looks mostly fine." | "ESCALATE. Evidence: `deliver()` is synchronous with no timeout/retry/isolation, so a slow merchant blocks `posting.post()` and a retry double-delivers `transfer.posted`. Accepted risk: none — this touches settled money. Confidence: high." |

The left column restates what each stage said and stops. The right column names a decision, ties it to concrete evidence from the pipeline's own artifacts, states what risk (if any) is being accepted, and gives an explicit confidence level. If your `synthesizer.md` could produce the left column, it isn't finished yet.


## 2 · Your exercise

Work through these steps in order:

1. Copy `reference/m5/explorer.md` and `reference/m5/critic.md` into `.claude/agents/` — these are completed, read-only agents, so you can run the whole team without having built them yourself in Module 3.
2. Build `.claude/commands/run-pipeline.md`: a slash command that orchestrates explorer → implementer → tester → critic → synthesizer in order, passing each stage's artifact forward as the next stage's input. The command body must encode the disagreement-rule pseudo-spec above verbatim in logic: if the critic raises a blocker, the implementer gets exactly one revision loop, and if the blocker still stands after that loop, the pipeline escalates instead of looping again or shipping anyway. It must also encode a low-confidence halt: if the synthesizer's own confidence is low, the pipeline stops and reports that rather than forcing a decision.
3. Complete `.claude/agents/synthesizer.md` from the `starter-agents/synthesizer.skeleton.md` skeleton. It stays read-only (`Read, Glob` only — no `Write`, no `Edit`, no `Bash`). Its output must state a recommendation (SHIP, REVISE, or ESCALATE), the evidence for it drawn from the pipeline's artifacts, the unresolved risks it is explicitly choosing to accept, and an explicit confidence statement — including what would raise that confidence.
4. Run the pipeline end to end on `scenarios/m5-webhooks.md`, writing the final recommendation to `artifacts/m5/synthesis.md`.


### What good looks like

A good run surfaces the webhook ordering blocker described in the grounding above — not as a vague "webhooks are risky" note, but as the specific failure mode: a slow or failing merchant endpoint blocks or fails the posting, or double-delivers `transfer.posted` on retry. `artifacts/m5/synthesis.md` states a clear decision and an explicit confidence level, and reading it should tell you something reading the raw upstream artifacts with `cat artifacts/m5/*.md` would not — that's the difference between synthesis and concatenation (see the side-by-side example above).

Given the real defect here, a well-built pipeline should ESCALATE rather than SHIP. A stall with a clear status — "blocked, here's why, here's what would resolve it" — beats a forced green run every time. If your pipeline ships this ticket without isolating delivery from the posting path, that's a sign the disagreement rule or the synthesizer's honesty didn't hold, not a sign the exercise is done.

Common failure modes:
- A synthesizer that averages the team's opinions into a vague "mostly good, some concerns" instead of choosing SHIP, REVISE, or ESCALATE.
- A `run-pipeline.md` that loops the critic and implementer indefinitely instead of escalating after one revision.
- A synthesis that states a decision but no confidence, or states confidence without evidence for it.


### Verify your work

This checklist is advisory, not a gate — it reflects back what it finds in `.claude/commands/run-pipeline.md`, `.claude/agents/synthesizer.md`, and `artifacts/m5/synthesis.md`. It's safe to run at any point, including before you've built anything. Note it requires a decision word **and** explicit confidence to both be present in `synthesis.md` — a synthesis that only mentions "escalate" in passing, without a stated confidence, does not pass.


In [ ]:
import pathlib
cmd = pathlib.Path(proj) / ".claude" / "commands" / "run-pipeline.md"
syn = pathlib.Path(proj) / ".claude" / "agents" / "synthesizer.md"
print(f"[{'x' if cmd.exists() else ' '}] run-pipeline.md exists")
if cmd.exists():
    c = cmd.read_text().lower()
    print(f"  [{'x' if 'escalate' in c else ' '}] encodes an escalate/disagreement rule?")
    print(f"  [{'x' if 'confidence' in c or 'halt' in c else ' '}] encodes a low-confidence halt?")
print(f"[{'x' if syn.exists() else ' '}] synthesizer.md exists")
out = pathlib.Path(proj) / "artifacts" / "m5" / "synthesis.md"
if out.exists():
    t = out.read_text().lower()
    print(f"[x] synthesis.md present ({len(t.split())} words)")
    has_decision = any(w in t for w in ("ship", "revise", "escalate"))
    has_confidence = "confidence" in t
    # REQUIRED: a bare "escalate" mention must NOT pass on its own. Both a
    # decision word AND an explicit confidence signal must be co-present.
    print(f"  [{'x' if has_decision else ' '}] states a decision (SHIP / REVISE / ESCALATE)?")
    print(f"  [{'x' if has_confidence else ' '}] states explicit confidence?")
    print(f"  [{'x' if (has_decision and has_confidence) else ' '}] BOTH co-present (this is what actually passes)?")
else:
    print("[ ] artifacts/m5/synthesis.md missing")


## 3 · Debrief

- Why does the disagreement rule cap the revision loop at one, instead of letting the critic and implementer iterate until the critic is satisfied?
- What's the difference between a synthesizer that "summarizes" the team's outputs and one that "decides"? Where in your `synthesizer.md` does that show up concretely?
- For this ticket specifically, what would have to be true of the implementation for ESCALATE to become SHIP?


---
### Take-home

An agent team is only as good as its orchestration rules: what happens on disagreement, and what happens on low confidence. Without both rules written down, a pipeline either loops forever or quietly forces green. The synthesizer's job is to decide, with stated evidence, accepted risk, and confidence — and a stall with a clear status is a legitimate, honest outcome, not a failure of the exercise.

(Related concept: Module 6 closes the loop with tests generated from acceptance criteria rather than from the implementation itself.)
